## Import thư viện

In [1]:
import os
import json
import re
import unicodedata
import pandas as pd

from tqdm import tqdm
from underthesea import word_tokenize

tqdm.pandas()

## 2. Khai báo đường dẫn

In [2]:
DATA_PATH = "../data/raw/news_dataset.json"

OUTPUT_DIR = "../data/processed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

OUTPUT_PKL = os.path.join(OUTPUT_DIR, "news_processed.pkl")
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "news_processed.csv")

## 3. Đọc dữ liệu gốc

In [3]:
with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

df.head()

Shape: (184539, 10)
Columns: ['id', 'author', 'content', 'picture_count', 'processed', 'source', 'title', 'topic', 'url', 'crawled_at']


,id,author,content,picture_count,processed,source,title,topic,url,crawled_at
0,218270,,"Chiều 31/7, Công an tỉnh Thừa Thiên - Huế đã c...",3,0,docbao.vn,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",Pháp luật,https://docbao.vn/phap-luat/ten-cuop-tiem-vang...,2022-08-01 09:09:22.817308
1,218269,(Nguồn: Sina),"Gần đây, Thứ trưởng Bộ Phát triển Kỹ thuật số,...",1,0,vtc.vn,"Bỏ qua mạng 5G, Nga tiến thẳng từ 4G lên 6G",Sống kết nối,https://vtc.vn/bo-qua-mang-5g-nga-tien-thang-t...,2022-08-01 09:09:21.181469
2,218268,Hồ Sỹ Anh,Kết quả thi tốt nghiệp THPT năm 2022 cho thấy ...,3,0,thanhnien.vn,Địa phương nào đứng đầu cả nước tổng điểm 3 mô...,Giáo dục,https://thanhnien.vn/dia-phuong-nao-dung-dau-c...,2022-08-01 09:09:15.311901
3,218267,Ngọc Ánh,Thống đốc Kentucky Andy Beshear hôm 31/7 cho h...,1,0,vnexpress,Người chết trong mưa lũ 'nghìn năm có một' ở M...,Thế giới,https://vnexpress.net/nguoi-chet-trong-mua-lu-...,2022-08-01 09:09:02.211498
4,218266,HẢI YẾN - MINH LÝ,Vụ tai nạn giao thông liên hoàn trên phố đi bộ...,12,0,soha,"Hải Phòng: Hình ảnh xe ""điên"" gây tai nạn liên...",Thời sự - Xã hội,https://soha.vn/hai-phong-hinh-anh-xe-dien-gay...,2022-08-01 09:09:01.601170


## 4. Chuẩn hóa tên cột

In [4]:
df.columns = [col.strip().lower() for col in df.columns]

print("Columns after normalize:")
print(df.columns.tolist())

Columns after normalize:
['id', 'author', 'content', 'picture_count', 'processed', 'source', 'title', 'topic', 'url', 'crawled_at']


## 5. Chọn các cột cần thiết

In [5]:
required_columns = [
    "id",
    "title",
    "content",
    "topic",
    "source",
    "url"
]

for col in required_columns:
    if col not in df.columns:
        df[col] = ""

for col in required_columns:
    df[col] = df[col].fillna("").astype(str)

df = df[required_columns].copy()

df.head()

,id,title,content,topic,source,url
0,218270,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...","Chiều 31/7, Công an tỉnh Thừa Thiên - Huế đã c...",Pháp luật,docbao.vn,https://docbao.vn/phap-luat/ten-cuop-tiem-vang...
1,218269,"Bỏ qua mạng 5G, Nga tiến thẳng từ 4G lên 6G","Gần đây, Thứ trưởng Bộ Phát triển Kỹ thuật số,...",Sống kết nối,vtc.vn,https://vtc.vn/bo-qua-mang-5g-nga-tien-thang-t...
2,218268,Địa phương nào đứng đầu cả nước tổng điểm 3 mô...,Kết quả thi tốt nghiệp THPT năm 2022 cho thấy ...,Giáo dục,thanhnien.vn,https://thanhnien.vn/dia-phuong-nao-dung-dau-c...
3,218267,Người chết trong mưa lũ 'nghìn năm có một' ở M...,Thống đốc Kentucky Andy Beshear hôm 31/7 cho h...,Thế giới,vnexpress,https://vnexpress.net/nguoi-chet-trong-mua-lu-...
4,218266,"Hải Phòng: Hình ảnh xe ""điên"" gây tai nạn liên...",Vụ tai nạn giao thông liên hoàn trên phố đi bộ...,Thời sự - Xã hội,soha,https://soha.vn/hai-phong-hinh-anh-xe-dien-gay...


## 6. Xóa bài không có nội dung

In [6]:
before = len(df)

df = df[
    (df["title"].str.strip() != "") |
    (df["content"].str.strip() != "")
].copy()

df = df.reset_index(drop=True)

after = len(df)

print("Before:", before)
print("After :", after)
print("Removed empty rows:", before - after)

Before: 184539
After : 184521
Removed empty rows: 18


## 7. Xóa dữ liệu trùng lặp

In [7]:
before = len(df)

df["duplicate_key"] = (
    df["title"].str.strip().str.lower() + " " +
    df["content"].str.strip().str.lower()
)

df = df.drop_duplicates(subset=["duplicate_key"]).copy()
df = df.drop(columns=["duplicate_key"])
df = df.reset_index(drop=True)

after = len(df)

print("Before:", before)
print("After :", after)
print("Removed duplicates:", before - after)

Before: 184521
After : 182349
Removed duplicates: 2172


## 8. Tạo raw_text

In [8]:
df["raw_text"] = (
    df["title"].astype(str) + " " +
    df["title"].astype(str) + " " +
    df["content"].astype(str)
)

df = df[df["raw_text"].str.strip() != ""].reset_index(drop=True)

print("Shape after creating raw_text:", df.shape)

df[["id", "title", "topic", "source", "raw_text"]].head()

Shape after creating raw_text: (182349, 7)


,id,title,topic,source,raw_text
0,218270,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",Pháp luật,docbao.vn,"Tên cướp tiệm vàng tại Huế là đại uý công an, ..."
1,218269,"Bỏ qua mạng 5G, Nga tiến thẳng từ 4G lên 6G",Sống kết nối,vtc.vn,"Bỏ qua mạng 5G, Nga tiến thẳng từ 4G lên 6G Bỏ..."
2,218268,Địa phương nào đứng đầu cả nước tổng điểm 3 mô...,Giáo dục,thanhnien.vn,Địa phương nào đứng đầu cả nước tổng điểm 3 mô...
3,218267,Người chết trong mưa lũ 'nghìn năm có một' ở M...,Thế giới,vnexpress,Người chết trong mưa lũ 'nghìn năm có một' ở M...
4,218266,"Hải Phòng: Hình ảnh xe ""điên"" gây tai nạn liên...",Thời sự - Xã hội,soha,"Hải Phòng: Hình ảnh xe ""điên"" gây tai nạn liên..."


## 9. Danh sách stopwords tiếng Việt

In [ ]:
vietnamese_stopwords = set([
    "là", "và", "của", "có", "cho", "với", "trong", "khi",
    "những", "các", "một", "được", "đã", "đang", "này",
    "đó", "thì", "mà", "ở", "về", "từ", "đến", "theo",
    "sau", "trước", "trên", "dưới", "vào", "ra",
    "cũng", "như", "nếu", "vì", "do", "để", "bị",
    "tại", "nên", "sẽ", "rằng", "nhiều", "ít",
    "hơn", "rất", "lại", "phải", "không"
])

## 10.Hàm làm sạch văn bản

In [11]:
def normalize_unicode(text):
    text = str(text)
    return unicodedata.normalize("NFC", text)


def clean_text(text):
    text = normalize_unicode(text)
    text = text.lower()

    # Xóa URL
    text = re.sub(r"http\S+|www\S+", " ", text)

    # Xóa email
    text = re.sub(r"\S+@\S+", " ", text)

    # Xóa HTML tag
    text = re.sub(r"<.*?>", " ", text)

    # Xóa xuống dòng, tab
    text = re.sub(r"[\n\r\t]", " ", text)

    # Giữ chữ tiếng Việt, chữ tiếng Anh, số và khoảng trắng
    text = re.sub(r"[^a-zA-ZÀ-ỹ0-9\s]", " ", text)

    # Chuẩn hóa khoảng trắng
    text = re.sub(r"\s+", " ", text).strip()

    return text


def remove_stopwords(text):
    tokens = text.split()
    tokens = [token for token in tokens if token not in vietnamese_stopwords]
    return " ".join(tokens)


def preprocess(text):
    text = clean_text(text)
    text = word_tokenize(text, format="text")
    text = remove_stopwords(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

## 11. Kiểm tra thử hàm tiền xử lý

In [ ]:
sample_text = df.loc[0, "raw_text"]

print("Before:")
print(sample_text[:1000])

print("\nAfter:")
print(preprocess(sample_text)[:1000])

## 12.Áp dụng tiền xử lý cho toàn bộ dataset

In [12]:
df["processed_text"] = df["raw_text"].progress_apply(preprocess)

df = df[df["processed_text"].str.strip() != ""].reset_index(drop=True)

print("Shape after preprocessing:", df.shape)

df[["id", "title", "topic", "processed_text"]].head()

100%|██████████| 182349/182349 [1:25:24<00:00, 35.58it/s]  


Shape after preprocessing: (182347, 8)


,id,title,topic,processed_text
0,218270,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",Pháp luật,tên cướp tiệm vàng huế đại_úy công_an công_tác...
1,218269,"Bỏ qua mạng 5G, Nga tiến thẳng từ 4G lên 6G",Sống kết nối,bỏ_qua mạng 5 g nga tiến thẳng 4 g lên 6 g bỏ_...
2,218268,Địa phương nào đứng đầu cả nước tổng điểm 3 mô...,Giáo dục,địa_phương nào đứng đầu cả nước tổng_điểm 3 mô...
3,218267,Người chết trong mưa lũ 'nghìn năm có một' ở M...,Thế giới,chết mưa_lũ nghìn mỹ tăng lên 28 chết mưa_lũ n...
4,218266,"Hải Phòng: Hình ảnh xe ""điên"" gây tai nạn liên...",Thời sự - Xã hội,hải_phòng hình_ảnh xe điên gây tai_nạn liên_ho...


## 13.Tạo tokens cho BM25

In [13]:
df["tokens"] = df["processed_text"].apply(lambda x: x.split())

df[["id", "title", "tokens"]].head()

,id,title,tokens
0,218270,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...","[tên, cướp, tiệm, vàng, huế, đại_úy, công_an, ..."
1,218269,"Bỏ qua mạng 5G, Nga tiến thẳng từ 4G lên 6G","[bỏ_qua, mạng, 5, g, nga, tiến, thẳng, 4, g, l..."
2,218268,Địa phương nào đứng đầu cả nước tổng điểm 3 mô...,"[địa_phương, nào, đứng, đầu, cả, nước, tổng_đi..."
3,218267,Người chết trong mưa lũ 'nghìn năm có một' ở M...,"[chết, mưa_lũ, nghìn, mỹ, tăng, lên, 28, chết,..."
4,218266,"Hải Phòng: Hình ảnh xe ""điên"" gây tai nạn liên...","[hải_phòng, hình_ảnh, xe, điên, gây, tai_nạn, ..."


## 14.Tạo token_count và lọc bài quá ngắn

In [14]:
df["token_count"] = df["tokens"].apply(len)

df["token_count"].describe()

count    182347.000000
mean        304.119541
std         244.755432
min           1.000000
25%         162.000000
50%         266.000000
75%         416.000000
max       38759.000000
Name: token_count, dtype: float64

In [15]:
before = len(df)

df = df[df["token_count"] >= 20].copy()
df = df.reset_index(drop=True)

after = len(df)

print("Before:", before)
print("After :", after)
print("Removed short documents:", before - after)

Before: 182347
After : 167624
Removed short documents: 14723


## 15.Tạo doc_id mới

In [16]:
df["doc_id"] = df.index

df[["doc_id", "id", "title", "topic", "source", "token_count"]].head()

,doc_id,id,title,topic,source,token_count
0,0,218270,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",Pháp luật,docbao.vn,363
1,1,218269,"Bỏ qua mạng 5G, Nga tiến thẳng từ 4G lên 6G",Sống kết nối,vtc.vn,202
2,2,218268,Địa phương nào đứng đầu cả nước tổng điểm 3 mô...,Giáo dục,thanhnien.vn,922
3,3,218267,Người chết trong mưa lũ 'nghìn năm có một' ở M...,Thế giới,vnexpress,187
4,4,218266,"Hải Phòng: Hình ảnh xe ""điên"" gây tai nạn liên...",Thời sự - Xã hội,soha,163


## 16.Kiểm tra dữ liệu sau xử lý

In [17]:
print("Final shape:", df.shape)

print("Columns:")
print(df.columns.tolist())

df[[
    "doc_id",
    "id",
    "title",
    "topic",
    "source",
    "url",
    "token_count",
    "processed_text"
]].head()

Final shape: (167624, 11)
Columns:
['id', 'title', 'content', 'topic', 'source', 'url', 'raw_text', 'processed_text', 'tokens', 'token_count', 'doc_id']


,doc_id,id,title,topic,source,url,token_count,processed_text
0,0,218270,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",Pháp luật,docbao.vn,https://docbao.vn/phap-luat/ten-cuop-tiem-vang...,363,tên cướp tiệm vàng huế đại_úy công_an công_tác...
1,1,218269,"Bỏ qua mạng 5G, Nga tiến thẳng từ 4G lên 6G",Sống kết nối,vtc.vn,https://vtc.vn/bo-qua-mang-5g-nga-tien-thang-t...,202,bỏ_qua mạng 5 g nga tiến thẳng 4 g lên 6 g bỏ_...
2,2,218268,Địa phương nào đứng đầu cả nước tổng điểm 3 mô...,Giáo dục,thanhnien.vn,https://thanhnien.vn/dia-phuong-nao-dung-dau-c...,922,địa_phương nào đứng đầu cả nước tổng_điểm 3 mô...
3,3,218267,Người chết trong mưa lũ 'nghìn năm có một' ở M...,Thế giới,vnexpress,https://vnexpress.net/nguoi-chet-trong-mua-lu-...,187,chết mưa_lũ nghìn mỹ tăng lên 28 chết mưa_lũ n...
4,4,218266,"Hải Phòng: Hình ảnh xe ""điên"" gây tai nạn liên...",Thời sự - Xã hội,soha,https://soha.vn/hai-phong-hinh-anh-xe-dien-gay...,163,hải_phòng hình_ảnh xe điên gây tai_nạn liên_ho...


## 17.Lưu dữ liệu đã xử lý

In [18]:
df.to_pickle(OUTPUT_PKL)

df.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig"
)

print("Saved pickle:", OUTPUT_PKL)
print("Saved csv   :", OUTPUT_CSV)

Saved pickle: ../data/processed\news_processed.pkl
Saved csv   : ../data/processed\news_processed.csv


## 18.Load lại kiểm tra

In [19]:
test_df = pd.read_pickle(OUTPUT_PKL)

print(test_df.shape)

test_df[[
    "doc_id",
    "id",
    "title",
    "topic",
    "source",
    "token_count",
    "processed_text"
]].head()

(167624, 11)


,doc_id,id,title,topic,source,token_count,processed_text
0,0,218270,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",Pháp luật,docbao.vn,363,tên cướp tiệm vàng huế đại_úy công_an công_tác...
1,1,218269,"Bỏ qua mạng 5G, Nga tiến thẳng từ 4G lên 6G",Sống kết nối,vtc.vn,202,bỏ_qua mạng 5 g nga tiến thẳng 4 g lên 6 g bỏ_...
2,2,218268,Địa phương nào đứng đầu cả nước tổng điểm 3 mô...,Giáo dục,thanhnien.vn,922,địa_phương nào đứng đầu cả nước tổng_điểm 3 mô...
3,3,218267,Người chết trong mưa lũ 'nghìn năm có một' ở M...,Thế giới,vnexpress,187,chết mưa_lũ nghìn mỹ tăng lên 28 chết mưa_lũ n...
4,4,218266,"Hải Phòng: Hình ảnh xe ""điên"" gây tai nạn liên...",Thời sự - Xã hội,soha,163,hải_phòng hình_ảnh xe điên gây tai_nạn liên_ho...
